In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os

for dirname, _, filenames in os.walk("./"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

./Project_6.ipynb
./.ipynb_checkpoints/Project_6-checkpoint.ipynb
./tetris_a-main/AI_Final_Project_Report.pdf
./tetris_a-main/README.md
./tetris_a-main/gpu_utils.py
./tetris_a-main/requirements.txt
./tetris_a-main/.ipynb_checkpoints/requirements-checkpoint.txt
./tetris_a-main/.ipynb_checkpoints/README-checkpoint.md
./tetris_a-main/src/mcts.py
./tetris_a-main/src/plots.ipynb
./tetris_a-main/src/greedy.py
./tetris_a-main/src/data_0.csv
./tetris_a-main/src/genetic_helpers.py
./tetris_a-main/src/plots2.ipynb
./tetris_a-main/src/analysis.ipynb
./tetris_a-main/src/game.py
./tetris_a-main/src/greedy_dropped_comparison.png
./tetris_a-main/src/main.py
./tetris_a-main/src/greedy_dropped.png
./tetris_a-main/src/randomChoice.py
./tetris_a-main/src/piece.py
./tetris_a-main/src/expect.png
./tetris_a-main/src/run_experiments.py
./tetris_a-main/src/genetic.py
./tetris_a-main/src/board.py
./tetris_a-main/src/analysis2.ipynb
./tetris_a-main/src/custom_model.py
./tetris_a-main/src/genetic_controller.py
.

# 1. Frame the problem
Using the customer description, Define the problem your trying to solve in your own words (remember this is not technial but must be specific so the customer understands the project

# 2. Get the Data 
Define how you recieved the data (provided, gathered..)

In [9]:
"""
Data was provided in the form of code files that created a playable game of Tetris when run stored in the current directory. There are also several examples of various types of
AI algorithms that have varying levels of success at playing the game, such as Greedy, Monte Carlo Tree Search, Genetic, and Random.
"""
import sys

print(sys.version)
print(sys.executable)

!{sys.executable} -m venv ~/tetris_env

%cd tetris_a
%pip install -r requirements.txt
!python main.py player

3.12.6 | packaged by conda-forge | (main, Sep 22 2024, 14:16:49) [GCC 13.3.0]
/opt/tljh/user/bin/python
[Errno 2] No such file or directory: 'tetris_a'
/home/jupyter-1005805/Project_6/tetris_a
Defaulting to user installation because normal site-packages is not writeable
Ignoring rubicon-objc: markers 'sys_platform == "darwin"' don't match your environment
Note: you may need to restart the kernel to use updated packages.
pygame 2.6.1 (SDL 2.28.4, Python 3.12.6)
Hello from the pygame community. https://www.pygame.org/contribute.html


# 3. Explore the Data
Gain insights into the data you have from step 2, making sure to identify any bias

In [28]:
"""
There are many existing models in the data provided, such as Greedy, Monte Carlo Tree Search, Genetic, and Random. The Greedy model is very successful and
runs for a very long time, while the other three models stop after only a few moves, with MCTS being the worst, then Genetic, then Random.
"""
import sys

!python src/main.py mcts
!python src/main.py genetic
!python src/main.py random

pygame 2.6.1 (SDL 2.28.4, Python 3.12.6)
Hello from the pygame community. https://www.pygame.org/contribute.html
1 0
2 0
3 0
4 0
5 0
6 0
7 0
8 0
8 0
pygame 2.6.1 (SDL 2.28.4, Python 3.12.6)
Hello from the pygame community. https://www.pygame.org/contribute.html
1 0
2 0
3 0
4 0
5 0
6 0
7 0
8 0
9 0
10 0
11 0
12 0
13 0
14 0
15 0
16 0
17 0
18 0
19 0
20 0
21 0
22 0
22 0
pygame 2.6.1 (SDL 2.28.4, Python 3.12.6)
Hello from the pygame community. https://www.pygame.org/contribute.html
IN Play i : 1 y: 0
best_x 1 best_piece <piece.Piece object at 0x7f9a8ba80560>
1 0
IN Play i : 1 y: 0
best_x 6 best_piece <piece.Piece object at 0x7f9a8ba80590>
2 0
IN Play i : 1 y: 3
best_x 1 best_piece <piece.Piece object at 0x7f99e9219430>
3 0
IN Play i : 1 y: 2
best_x 6 best_piece <piece.Piece object at 0x7f9a8ba80560>
4 0
IN Play i : 1 y: 5
best_x 0 best_piece <piece.Piece object at 0x7f99e9219dc0>
5 0
IN Play i : 1 y: 0
best_x 4 best_piece <piece.Piece object at 0x7f9a8ba80560>
6 0
OUT of PLAY
IN Play i : 2 y

# 4.Prepare the Data


Apply any data transformations and explain what and why


In [101]:
"""
I was able to convert the results from the random game into a
dataframe which I can use later on to train a model that can play
Tetris.
"""
import sys
import random
import numpy as np
import pandas as pd

if "src" not in sys.path:
    sys.path.append("src")

from game import Game

CANONICAL_SHAPES = {
    "I": ((0, 0), (0, 1), (0, 2), (0, 3)),
    "O": ((0, 0), (0, 1), (1, 0), (1, 1)),
    "T": ((0, 0), (1, 0), (1, 1), (2, 0)),
    "L1": ((0, 0), (0, 1), (0, 2), (1, 0)),
    "L2": ((0, 0), (1, 0), (1, 1), (1, 2)),
    "S1": ((0, 0), (1, 0), (1, 1), (2, 1)),
    "S2": ((0, 1), (1, 0), (1, 1), (2, 0))
}

def normalize(coords):
    min_x = min(x for x, y in coords)
    min_y = min(y for x, y in coords)

    return tuple(sorted(
        (x - min_x, y - min_y)
        for x, y in coords
    ))

def identify_piece(piece):
    target = normalize(piece.body)

    for name, shape in CANONICAL_SHAPES.items():
        current = normalize(shape)

        for rotation in range(4):
            if current == target:
                return name, rotation

            current = normalize([(-y, x) for x, y in current])

    return "Unknown", -1

def extract_features(board, piece):
    heights = np.array(board.heights, dtype=np.float32)
    holes = []

    for col in range(board.width):
        found_block = False
        count = 0

        for row in range(board.height):
            if board.board[row][col]:
                found_block = True
            elif found_block:
                count += 1

        holes.append(count)
    
    holes = np.array(holes)
    aggregate_height = heights.sum()
    max_height = heights.max()
    bumpiness = np.abs(np.diff(heights)).sum()

    piece_id = {
        "I": 0,
        "O": 1,
        "T": 2,
        "L1": 3,
        "L2": 4,
        "S1": 5,
        "S2": 6
    }

    shape, rotation = identify_piece(piece)
    piece_features = np.zeros(7)

    if shape != "Unknown":
        piece_features[piece_id[shape]] = 1

    features = np.concatenate([
        heights / 20,
        holes / 20,
        np.array([
            aggregate_height / 200,
            max_height / 20,
            bumpiness / 180
        ]),
        piece_features
    ])

    return features.astype(np.float32)

def calculate_potential(board):
    heights = np.array(board.heights)
    holes = 0

    for col in range(board.width):
        found = False

        for row in range(board.height):
            if board.board[row][col]:
                found = True
            elif found:
                holes += 1

    aggregate = heights.sum()
    bumpiness = np.abs(np.diff(heights)).sum()

    return (
        -0.5 * aggregate
        -3.0 * holes
        -0.2 * bumpiness
    )

def count_occupied(board):

    return sum(
        sum(bool(cell) for cell in row)
        for row in board.board
    )

def collect_random_games(n_games=200):
    dataset = []

    for game_id in range(n_games):
        game = Game("random")
        piece_number = 0

        while True:
            current_piece = game.curr_piece

            state = extract_features(game.board, current_piece)
            shape, rotation = identify_piece(current_piece)
            old_potential = calculate_potential(game.board)
            old_occupied = count_occupied(game.board)

            # Use the existing random agent
            x, selected_piece = game.ai.get_best_move(game.board, current_piece)
            y = game.board.drop_height(selected_piece, x)

            game.curr_piece = selected_piece
            game.drop(y, x=x)

            done = game.board.top_filled()
            new_occupied = count_occupied(game.board)

            lines = round((old_occupied + 4 - new_occupied) / game.board.width)
            lines = max(0, min(4, lines))

            new_potential = calculate_potential(game.board)
            reward = (10 * lines + 0.1 * (new_potential - old_potential))

            if done:
                reward -= 20

            next_state = (
                np.zeros(30, dtype=np.float32)
                if done
                else extract_features(
                    game.board,
                    game.curr_piece
                )
            )

            row = {
                "game": game_id,
                "piece_number": piece_number,
                "shape": shape,
                "rotation": rotation,
                "x": x,
                "drop_y": y,
                "reward": reward,
                "lines_cleared": lines,
                "game_over": done
            }

            for i, value in enumerate(state):
                row[f"state_{i}"] = value

            for i, value in enumerate(next_state):
                row[f"next_state_{i}"] = value

            dataset.append(row)
            piece_number += 1

            if done:
                break

        if (game_id + 1) % 10 == 0:
            print(f"Completed game {game_id + 1}/{n_games}")

    return pd.DataFrame(dataset)

random.seed(42)
np.random.seed(42)

df = collect_random_games(200)
df.to_csv("random_tetris_dataset.csv", index=False)

print("\nDataset generated successfully!")
print("Shape:", df.shape)

IN Play i : 1 y: 0
best_x 4 best_piece Piece(I, 0)
IN Play i : 1 y: 4
best_x 2 best_piece Piece(L2, 1)
IN Play i : 1 y: 0
best_x 9 best_piece Piece(I, 0)
IN Play i : 1 y: 0
best_x 0 best_piece Piece(O, 0)
IN Play i : 1 y: 6
best_x 3 best_piece Piece(I, 1)
OUT of PLAY
IN Play i : 2 y: 7
best_x 3 best_piece Piece(T, 0)
IN Play i : 1 y: 9
best_x 3 best_piece Piece(T, 1)
IN Play i : 1 y: 2
best_x 0 best_piece Piece(O, 0)
IN Play i : 1 y: 8
best_x 5 best_piece Piece(L1, 1)
IN Play i : 1 y: 12
best_x 3 best_piece Piece(S1, 1)
IN Play i : 1 y: 15
best_x 1 best_piece Piece(S2, 0)
IN Play i : 1 y: 9
best_x 5 best_piece Piece(O, 0)
IN Play i : 1 y: 16
best_x 0 best_piece Piece(S2, 1)
IN Play i : 1 y: 11
best_x 6 best_piece Piece(O, 0)
OUT of PLAY
IN Play i : 2 y: 13
best_x 5 best_piece Piece(I, 1)
IN Play i : 1 y: 18
best_x 1 best_piece Piece(T, 1)
IN Play i : 1 y: 0
best_x 1 best_piece Piece(L2, 1)
IN Play i : 1 y: 0
best_x 6 best_piece Piece(L2, 0)
IN Play i : 1 y: 1
best_x 5 best_piece Piece(

# 5. Model the data
Using selected ML models, experment with your choices and describe your findings. Finish by selecting a Model to continue with


In [162]:
"""
"""
import os
import shutil
import numpy as np
import pandas as pd
import torch
from datetime import datetime
from agent import DQNAgent
from environment import TetrisEnvrionment

EPISODES = 100000
SAVE_INTERVAL = 250
EVAL_INTERVAL = 100
EVAL_GAMES = 5

CHECKPOINT_PATH = "saved_models/dqn_checkpoint.pth"
TRAINING_CSV = "training_results.csv"
EVALUATION_CSV = "evaluation_results.csv"

# The last confirmed checkpoint corresponds to episode 87,000.
CHECKPOINT_EPISODE = 87000

os.makedirs("saved_models", exist_ok=True)
backup_folder = "resume_backup_" + datetime.now().strftime("%Y%m%d_%H%M%S")
os.makedirs(backup_folder, exist_ok=True)

for path in [CHECKPOINT_PATH, TRAINING_CSV, EVALUATION_CSV]:
    if os.path.exists(path):
        shutil.copy2(path, os.path.join(backup_folder, os.path.basename(path)))

print("Original files backed up.")

env = TetrisEnvironment()
agent = DQNAgent()

# Restore the saved neural network and optimizer.
agent.load(CHECKPOINT_PATH)

print("\nCheckpoint loaded successfully!")
print("Training steps:", agent.steps)
print("Epsilon:", agent.epsilon)

if not os.path.exists(TRAINING_CSV):
    raise FileNotFoundError(TRAINING_CSV)

training_df = pd.read_csv(TRAINING_CSV)

last_training_episode = int(training_df["episode"].max())

if last_training_episode != CHECKPOINT_EPISODE:
    raise ValueError(
        f"Expected training CSV to end at episode "
        f"{CHECKPOINT_EPISODE}, but found "
        f"{last_training_episode}. Check checkpoint alignment."
    )

resume_episode = CHECKPOINT_EPISODE
training_results = training_df.to_dict("records")

if os.path.exists(EVALUATION_CSV):
    evaluation_df = pd.read_csv(EVALUATION_CSV)
    evaluation_df = evaluation_df[evaluation_df["episode"] <= resume_episode]
    evaluation_results = evaluation_df.to_dict("records")
else:
    evaluation_results = []

print("\nResuming training:")
print("Starting episode:", resume_episode + 1)
print("Remaining episodes:", EPISODES - resume_episode)

def evaluate_agent(agent, games=5):
    original_epsilon = agent.epsilon
    agent.epsilon = 0.0

    scores = []
    eval_env = TetrisEnvironment()

    for game_number in range(games):
        state = eval_env.reset()
        done = False

        while not done:
            valid_actions = eval_env.get_valid_actions()

            if not valid_actions:
                break

            action = agent.select_action(state, valid_actions)
            next_state, reward, done, info = eval_env.step(action)
            state = next_state

        scores.append({
            "pieces": eval_env.game.pieces_dropped,
            "lines": eval_env.game.rows_cleared
        })

    agent.epsilon = original_epsilon
    average_lines = np.mean([s["lines"] for s in scores])
    average_pieces = np.mean([s["pieces"] for s in scores])

    return average_lines, average_pieces

for episode in range(resume_episode, EPISODES):
    episode_number = episode + 1

    state = env.reset()
    episode_reward = 0
    done = False
    loss = None

    while not done:
        valid_actions = env.get_valid_actions()

        if not valid_actions:
            break

        action = agent.select_action(state, valid_actions)
        next_state, reward, done, info = env.step(action)

        if not done:
            next_valid_actions = env.get_valid_actions()

            if not next_valid_actions:
                done = True
        else:
            next_valid_actions = []

        agent.remember(state, action, reward, next_state, done, next_valid_actions)
        
        loss = agent.learn()
        state = next_state
        episode_reward += reward

    training_results.append({
        "episode": episode_number,
        "pieces": env.game.pieces_dropped,
        "lines": env.game.rows_cleared,
        "reward": episode_reward,
        "epsilon": agent.epsilon,
        "loss": loss if loss is not None else 0
    })

    if episode_number % 100 == 0:
        recent = training_results[-10:]

        avg_lines = np.mean([r["lines"] for r in recent])
        avg_pieces = np.mean([r["pieces"] for r in recent])
        avg_reward = np.mean([r["reward"] for r in recent])

        print(
            f"Episode {episode_number}/{EPISODES} | "
            f"Lines: {avg_lines:.2f} | "
            f"Pieces: {avg_pieces:.2f} | "
            f"Reward: {avg_reward:.2f} | "
            f"Epsilon: {agent.epsilon:.4f}"
        )

    if episode_number % EVAL_INTERVAL == 0:
        avg_lines, avg_pieces = evaluate_agent(agent, games=EVAL_GAMES)

        evaluation_results.append({
            "episode": episode_number,
            "average_lines": avg_lines,
            "average_pieces": avg_pieces
        })

        pd.DataFrame(evaluation_results).to_csv(EVALUATION_CSV, index=False)

    if episode_number % SAVE_INTERVAL == 0:
        agent.save(CHECKPOINT_PATH)
        pd.DataFrame(training_results).to_csv(TRAINING_CSV, index=False)
        print(f"Checkpoint saved at episode {episode_number}")

agent.save(CHECKPOINT_PATH)
agent.save("saved_models/dqn_final.pth")

pd.DataFrame(training_results).to_csv(TRAINING_CSV, index=False)
pd.DataFrame(evaluation_results).to_csv(EVALUATION_CSV, index=False)

print("\nTraining completed!")
print("Final model saved.")

Original files backed up.

Checkpoint loaded successfully!
Training steps: 4336553
Epsilon: 0.05

Resuming training:
Starting episode: 87001
Remaining episodes: 13000
Episode 87100/100000 | Lines: 5.20 | Pieces: 43.60 | Reward: 20.09 | Epsilon: 0.0500
Episode 87200/100000 | Lines: 6.00 | Pieces: 47.70 | Reward: 28.61 | Epsilon: 0.0500
Checkpoint saved at episode 87250
Episode 87300/100000 | Lines: 7.00 | Pieces: 50.80 | Reward: 38.22 | Epsilon: 0.0500
Episode 87400/100000 | Lines: 9.40 | Pieces: 57.10 | Reward: 62.54 | Epsilon: 0.0500
Episode 87500/100000 | Lines: 8.70 | Pieces: 51.80 | Reward: 54.85 | Epsilon: 0.0500
Checkpoint saved at episode 87500
Episode 87600/100000 | Lines: 5.70 | Pieces: 46.20 | Reward: 25.61 | Epsilon: 0.0500
Episode 87700/100000 | Lines: 10.90 | Pieces: 60.60 | Reward: 77.37 | Epsilon: 0.0500
Checkpoint saved at episode 87750
Episode 87800/100000 | Lines: 6.30 | Pieces: 47.40 | Reward: 31.45 | Epsilon: 0.0500
Episode 87900/100000 | Lines: 10.60 | Pieces: 57.6

# 6. Fine Tune the Model

With the select model descibe the steps taken to acheve the best rusults possiable 


In [95]:
"""
After increasing the amount of epochs, the accuracy increased to 26.47%.
"""


'\nAfter increasing the amount of epochs, the accuracy increased to 26.47%.\n'

# 7. Present
In a customer faceing Document provide summery of finding and detail approach taken


# 8. Launch the Model System
Define your production run code, This should be self susficent and require only your model pramaters 


In [11]:
from game import Game

def inference():
    game = Game("student")
    game.run()

inference()

error: XDG_RUNTIME_DIR not set in the environment.
ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5182:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5182:(_snd_config_evaluate) function snd_func_concat returned error: No such file or directory
ALSA lib confmisc.c:1334:(snd_func_refer) error evaluating name
ALSA lib conf.c:5182:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5705:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2664:(snd_pcm_open_noupdate) Unknown PCM default


AI
Pieces dropped: 101
Rows cleared: 28


In [ ]:
!cd Project_6
!git init
!git add Project_6/Project_6.ipynb
!git commit -m "Initial commit"
!git branch -M main
!git config --global user.name "AldreiC"
!git config --global user.email "pl1005805@ahschool.com"
!git push origin main

Reinitialized existing Git repository in /home/jupyter-1005805/.git/
On branch main
Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.bash_logout
	.bashrc
	.cache/
	.config/
	.gitconfig
	.ipython/
	.jupyter/
	.keras/
	.local/
	.profile
	Project_1/
	Project_2/
	Project_3/
	Project_4/
	Project_5/
	Project_6/.ipynb_checkpoints/
	Project_6/tetris_a/
	Project_6/tetris_env/

nothing added to commit but untracked files present (use "git add" to track)
Username for 'https://github.com': 